In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import(
    accuracy_score,classification_report,confusion_matrix,ConfusionMatrixDisplay
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ALPHA = 0.01
POPULATION_SIZE = 15
MAX_ITER = 40
K_MIN,K_MAX = 1,15
N_FEATURES = 13

print(f"✅Library Imported Successfull")
print(f"Random seed       : {RANDOM_SEED}")
print(f"Population Size   : {POPULATION_SIZE}")
print(f"Max iterations    : {MAX_ITER}")
print(f"K_Neighbour range : [{K_MIN}, {K_MAX}]")
print(f"Penalty alpha     : {ALPHA}")

In [ ]:
# Data Preprocessing

# X_train = Training data
# y_train = Output for Training data
# X_test = Testing data
# y_test = Output for Testing data

def load_data(test_size: float=0.3, random_state: int = RANDOM_SEED):
    wine = load_wine()
    X, y = wine.data, wine.target
    feature_names = list(wine.feature_names)
    class_names   = list(wine.target_names)

    X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=test_size,random_state=random_state,stratify=y)
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test) 

    return X_train, X_test, y_train, y_test, feature_names, class_names


X_train, X_test, y_train, y_test, FEATURE_NAMES, CLASS_NAMES = load_data(test_size=0.3)


print(f"Total Sample : 178")
print(f"Training Set : {X_train.shape[0]} Samples, {X_train.shape[1]} Features")
print(f"Testing Set  : {X_test.shape[0]} Sampeles")
print(f"Classes Names: {CLASS_NAMES}")
print(f"FEATURE_NAMES: {FEATURE_NAMES}")

In [ ]:
# Baseline KNN Model

# X_train = Input Data
# y_train = output data for input
def train_knn(X_train, y_train, k:int = 5, feature_mask: np.ndarray = None)-> KNeighborsClassifier:
    if feature_mask is not None:
        mask = feature_mask.astype(bool)
        if mask.sum()==0:
            mask[0]=True
        X_train = X_train[:,mask]
        
    classifier = KNeighborsClassifier(n_neighbors=int(k))
    classifier.fit(X_train,y_train)    # Learn on input and output
    return classifier



def evaluate_model(classifier: KNeighborsClassifier, X_test,y_test,feature_mask: np.ndarray = None)->float:
    # X_test = Testing data
    # y_test = output for Testing data

    if feature_mask is not None:
        mask = feature_mask.astype(bool)
        if mask.sum()==0:
            mask[0]=True
        X_test = X_test[:,mask]
    
    prediction = classifier.predict(X_test)
    prediction_accuracy = accuracy_score(y_test,prediction)
    return prediction_accuracy




def fitness_function(solution:np.ndarray, X_train, y_train):
    k = int(np.clip(round(solution[0]), K_MIN, K_MAX))
    feature_mask = (solution[1:] >= 0.5).astype(int)

    if feature_mask.sum() == 0:
        feature_mask[0] = 1

    X_selected = X_train[:, feature_mask.astype(bool)]

    model = KNeighborsClassifier(n_neighbors=k)

    scores = cross_val_score(model, X_selected, y_train, cv=5)
    accuracy = scores.mean()
    penalty = feature_mask.sum()/N_FEATURES

    return accuracy - penalty*ALPHA


baseline_model = train_knn(X_train,y_train,k=5,feature_mask=None)
baseline_accuracy = evaluate_model(baseline_model,X_test,y_test,feature_mask=None)

baseline_cross_validate_score = cross_val_score(KNeighborsClassifier(n_neighbors=5), X_train, y_train, cv=5)
# Mean of Accuracy on 5 Different validation

print("BASELINE KNN RESULT")
print(f"-"*50)
print(f"K (Neighbour)     : 5")
print(f"Feature Used      : All {N_FEATURES}")
print(f"Baseline Accuracy : {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")
print(f"5 Fold CV Accuracy: {baseline_cross_validate_score.mean():.4f} ± {baseline_cross_validate_score.std():.4f}")
print(f"-"*50)

In [ ]:
# Whale Optimization Model (From Scratch)

def whale_optimization(X_train,y_train,population_size: int=POPULATION_SIZE, max_iteration: int=MAX_ITER, random_state: int=RANDOM_SEED):

    rng = np.random.default_rng(random_state)
    dim = N_FEATURES+1

    lb = np.array([K_MIN] + [0.0]*N_FEATURES)
    ub = np.array([K_MAX] + [1.0]*N_FEATURES)

    position = lb + rng.random((population_size,dim)) * (ub-lb)

    initial_fitness = np.array([fitness_function(position[i],X_train,y_train) for i in range(population_size)])
    

    best_idx = np.argmax(initial_fitness)
    best_position = position[best_idx].copy()
    best_fitness_score = initial_fitness[best_idx]

    convergence = [best_fitness_score]

    # Whale Optimization Algorithm
    for t in range(max_iteration):
        a = 2.0 - t*(2.0/max_iteration)
        
        for i in range(population_size):
            r1,r2 = rng.random(), rng.random()
            A = 2.0 * a * r1 - a    # Movement Contoller(Exploration or exploiation)
            C = 2.0 * r2            # Distance Emphesis(How whales moves, same whale moves different distance becasue of it)
            b = 1.0                 # Spiral Shape Constant(How aggressive whale approach solution)
            l = rng.uniform(-1,1)   # Each whale takes different path in spiral to reach solution
            p = rng.random()        # p>0.5 then spiral, p<0.5 then encircling

            if p<0.5:
                if abs(A)<1:
                    # Encircling(Exploitation)
                    D = abs(C * best_position-position[i])
                    position[i] = best_position - A * D
                else: 
                    # Random Search(Exploration)
                    rnd_idx = rng.integers(0,population_size)
                    rnd_whale = position[rnd_idx]
                    D = abs(C * rnd_whale - position[i])
                    position[i] = rnd_whale - A * D
            else:
                # Bubble net spiral update
                D_leader = abs(best_position - position[i])
                position[i] = (D_leader * np.exp(b * l) * np.cos(2 * np.pi * l) + best_position)
            
            position[i] = np.clip(position[i],lb,ub)
            
            fit = fitness_function(position[i],X_train,y_train)                

            if fit > best_fitness_score:
                best_fitness_score = fit
                best_position = position[i].copy()

        
        convergence.append(best_fitness_score)

        if (t+1)%10==0 or t==0:
            print(f"WOA Iter: {t+1:3}/40  |  Best Fitness = {best_fitness_score:.4f}")
    
    return best_position,best_fitness_score,convergence


print(f"Running Whale Optimization Algorithm\n{'-'*50}")


woa_solution, woa_train_accuracy, woa_convergence = whale_optimization(X_train,y_train,POPULATION_SIZE,MAX_ITER,RANDOM_SEED)

woa_best_k = int(np.clip(round(woa_solution[0]),K_MIN,K_MAX))
woa_feature_mask = (woa_solution[1:]>=0.5).astype(int)
woa_features_names = [FEATURE_NAMES[i] for i,j in enumerate(woa_feature_mask) if j==1]


woa_model = train_knn(X_train, y_train, woa_best_k, woa_feature_mask)
woa_test_accuracy = evaluate_model(woa_model,X_test,y_test,woa_feature_mask)


print(f"{'-'*50}\n")
print("✅ WOA COMPLETED!")
print('='*50)
print(f"WOA RESULTS\n{'-'*13}")
print(f"Best K-Neighbour      : {woa_best_k}")
print(f"Total Feature Selected: {len(woa_features_names)}/{N_FEATURES}")
print(f"Selected Features     : {woa_features_names}\n")
print(f"WOA Test Accuracy     : {woa_test_accuracy:.4f} ({woa_test_accuracy*100.0:.2f}%)")
print(f"WOA CV Accuracy       : {woa_train_accuracy:.4f} ({woa_train_accuracy*100.0:.2f}%)")


In [ ]:
# Genetic Algorithm

def _init_population_ga(population_size: int, rng)->np.ndarray:
    population = np.zeros((population_size,1+N_FEATURES),dtype=float)
    population[:,0] = rng.integers(K_MIN,K_MAX+1,size=population_size).astype(float)
    population[:,1:] = rng.integers(0,2,size=(population_size,N_FEATURES)).astype(float)
    return population


def _tournament_selection(population:np.ndarray, fitnesses: np.ndarray, tournament_size: int, rng)->np.ndarray:
    contender = rng.choice(len(population),size=tournament_size,replace=False)
    winner = contender[np.argmax(fitnesses[contender])]
    return population[winner].copy()


def _single_point_crossover(parent1:np.ndarray, parent2:np.ndarray, rng, crossover_rate: float=0.8):
    if rng.random()<crossover_rate:
        point = rng.integers(1,len(parent1))
        child1 = np.concatenate([parent1[:point],parent2[point:]])
        child2 = np.concatenate([parent2[:point],parent1[point:]])
        return child1, child2
    return parent1.copy(), parent2.copy()


def _mutate(chromosome:np.ndarray, rng, mutation_rate: float=0.05)->np.ndarray:
    chrom = chromosome.copy()

    if rng.random() < mutation_rate:
        delta = rng.choice([-1,1])
        chrom[0] = np.clip(chrom[0]+delta, K_MIN, K_MAX)
    
    for j in range(1,len(chrom)):
        if rng.random()<mutation_rate:
            chrom[j] = 1.0 - chrom[j]

    if chrom[1:].sum() == 0:
        chrom[rng.integers(1, len(chrom))] = 1.0

    return chrom


def genetic_algorithm(X_train, y_train, pop_size: int=POPULATION_SIZE, max_gen: int=MAX_ITER, tournament_size: int = 3, crossover_rate: float = 0.8, mutation_rate: float = 0.05, random_state: int = RANDOM_SEED):
    rng = np.random.default_rng(random_state)

    population = _init_population_ga(pop_size,rng)

    initial_fitness = np.array([fitness_function(population[i],X_train,y_train) for i in range(pop_size)])

    best_idx = np.argmax(initial_fitness)
    best_population = population[best_idx].copy()
    best_fitness_score = initial_fitness[best_idx]

    convergence = [best_fitness_score]

    for i in range(max_gen):
        new_population = [best_population.copy()]

        while len(new_population) < pop_size:
            parent1 = _tournament_selection(population, initial_fitness, tournament_size, rng)
            parent2 = _tournament_selection(population, initial_fitness, tournament_size, rng)
            
            child1, child2 = _single_point_crossover(parent1,parent2,rng,crossover_rate)
            
            child1 = _mutate(child1,rng,mutation_rate)
            child2 = _mutate(child2,rng,mutation_rate)

            new_population.append(child1)
            if len(new_population) < pop_size:
                new_population.append(child2)
            
        population = np.array(new_population)

        initial_fitness = np.array([fitness_function(population[i],X_train,y_train) for i in range(pop_size)])

        best_gen = np.argmax(initial_fitness)
        if initial_fitness[best_gen]>best_fitness_score:
            best_population = population[best_gen].copy()
            best_fitness_score = initial_fitness[best_gen]
        
        convergence.append(best_fitness_score)

        if (i+1) % 10 == 0  or i == 0:
            print(f"GA Generation: {i+1:3}/{max_gen}   |  Best Fitness Score = {best_fitness_score:.4f} ({best_fitness_score*100.0:.4f}%)")
        
    return best_population,best_fitness_score,convergence


print(f"Running Genetic Algorithm\n{'-'*50}")

ga_solution, ga_train_accuracy, ga_convergence = genetic_algorithm(X_train,y_train)     

ga_best_k = int(np.clip(round(ga_solution[0]),K_MIN,K_MAX))
ga_feature_mask = ga_solution[1:].astype(int)
if ga_feature_mask.sum() == 0:
    ga_feature_mask[0] = 1
ga_features_names = [FEATURE_NAMES[i] for i,j in enumerate(ga_feature_mask) if j==1]

ga_model = train_knn(X_train,y_train,ga_best_k,ga_feature_mask)
ga_test_accuracy = evaluate_model(ga_model,X_test,y_test,ga_feature_mask)

print(f"{'-'*50}\n")
print("✅ GA COMPLETED!")
print('='*50)
print(f"GA RESULTS\n{'-'*13}")
print(f"Best K-Neighbour      : {ga_best_k}")
print(f"Total Feature Selected: {len(ga_features_names)}/{N_FEATURES}")
print(f"Selected Features     : {ga_features_names}\n")
print(f"GA Test Accuracy     : {ga_test_accuracy:.4f} ({ga_test_accuracy*100.0:.2f}%)")
print(f"GA CV Accuracy       : {ga_train_accuracy:.4f} ({ga_train_accuracy*100.0:.2f}%)")

In [ ]:
# Detailed Classification Report

X_test_woa = X_test[:,woa_feature_mask.astype(bool)]
X_test_ga  = X_test[:, ga_feature_mask.astype(bool)]

y_pred_baseline = baseline_model.predict(X_test)
y_pred_woa = woa_model.predict(X_test_woa)
y_pred_ga = ga_model.predict(X_test_ga)

print("Detailed Classification Report - Baseline KNN")
print('-'*46)

print(classification_report(y_test,y_pred_baseline,target_names=CLASS_NAMES))

print("Detailed Classification Report - WOA-Optimized KNN")
print('-'*46)

print(classification_report(y_test,y_pred_woa,target_names=CLASS_NAMES))

print("Detailed Classification Report - GA-Optimized KNN")
print('-'*46)

print(classification_report(y_test,y_pred_ga,target_names=CLASS_NAMES))

In [ ]:
# Final Comparison and Analysis

method = ["Baseline KNN", "WOA-Optimized KNN", "GA-Optimization KNN"]

accuracies = [baseline_accuracy, woa_test_accuracy, ga_test_accuracy]
cross_validate_means = [baseline_cross_validate_score.mean(), woa_train_accuracy, ga_train_accuracy]
best_method = method[np.argmax(accuracies)]

print(f"📊 FINAL ACCURACY COMPARISON \n(Test data was not seen during optimization -- no data leakage)\n{'*'*65}")

print(f"\n{'Method':<23} {'Testing Accuracy':>17} {'Training Cross-Validation Accuracy':>41}")
print(f"{'-'*7:<23} {'-'*17:>18} {'-'*35:>41}")

for m,a,cv in zip(method,accuracies,cross_validate_means):
    marker = "  << BEST" if m == best_method else ""
    print(f"•{m:<23} {a:>8.4f} ({a*100.0:.2f})% {cv:>20.4f} ({cv*100.0:.2f})% {marker}")

print(f"\n{'*'*75}\n")

print(f"📈 Improvement over Baseline (Testing Accuracy):\n")
print(f"     •WOA-Optimization : {(woa_test_accuracy - baseline_accuracy)*100:+.2f}%")
print(f"     •GA-Optimization  : {(ga_test_accuracy  - baseline_accuracy)*100:+.2f}%")

winner = ["Baseline KNN (No Optimisation Needed)","Whale Optimization Algorithm","Genetic Algorithm Optimization"]
print(f"\nBest Method: * {winner[np.argmax(accuracies)]} *")
print(f"\n{'*'*65}\n")

woa_cv = cross_val_score(KNeighborsClassifier(n_neighbors=woa_best_k),X_train[:,woa_feature_mask.astype(bool)],y_train,cv=5)
ga_cv = cross_val_score(KNeighborsClassifier(n_neighbors=ga_best_k),X_train[:,ga_feature_mask.astype(bool)],y_train,cv=5)

print(f"5-Fold CV on Training Set Only (honest generalisation estimate):")
print('-'*65)
print(f"•Baseline  : {baseline_cross_validate_score.mean():.4f}% ± {baseline_cross_validate_score.std():.4f}%")
print(f"•WOA       : {woa_cv.mean():.4f}% ± {woa_cv.std():.4f}%")
print(f"•GA        : {ga_cv.mean():.4f}% ± {ga_cv.std():.4f}%")

In [ ]:
# Visualization

# WOA and GA Convergence graph

plt.style.use("seaborn-v0_8-whitegrid")
FIG_STYLE = dict(facecolor='#f8f9fa')

fig, axes = plt.subplots(1, 2, figsize=(14, 6), **FIG_STYLE)

ax = axes[0]
ax.plot(range(len(woa_convergence)), woa_convergence,color='#1565C0', linewidth=2.5, marker='o',markersize=4, markevery=5, label='WOA Best Accuracy')
ax.axhline(y=baseline_accuracy, color='grey', linestyle='--', linewidth=1.5, label=f'Baseline ({baseline_accuracy:.4f})')
ax.set_title('WOA Convergence Curve', fontsize=14, fontweight='bold')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Best Accuracy', fontsize=12)
ax.set_ylim([max(0, min(woa_convergence) - 0.05), 1.02])
ax.legend(fontsize=10)
ax.set_facecolor('#fdfdfd')

ax = axes[1]
ax.plot(range(len(ga_convergence)), ga_convergence, color='#2E7D32', linewidth=2.5, marker='s',markersize=4, markevery=5, label='GA Best Accuracy')
ax.axhline(y=baseline_accuracy, color='grey', linestyle='--', linewidth=1.5, label=f'Baseline ({baseline_accuracy:.4f})')
ax.set_title('GA Convergence Curve', fontsize=14, fontweight='bold')
ax.set_xlabel('Generation', fontsize=12)
ax.set_ylabel('Best Accuracy', fontsize=12)
ax.set_ylim([max(0, min(ga_convergence) - 0.05), 1.02])
ax.legend(fontsize=10)
ax.set_facecolor('#fdfdfd')

fig.suptitle('Optimization Convergence — WOA vs GA', fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('convergence_curves.png', dpi=1500, bbox_inches='tight')
plt.show()
print("Figure saved: convergence_curves.png")


In [ ]:
# Accuracy Bar Graph

fig, ax = plt.subplots(figsize=(10, 5), **FIG_STYLE)
bar_colors = ['#546E7A', "#1AEBEB", '#2E7D32']
bars = ax.bar(method, [a * 100 for a in accuracies], color=bar_colors, width=0.45, edgecolor='black', linewidth=0.6)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f'{acc*100:.2f}%',
            ha='center', va='bottom',
            fontsize=12, fontweight='bold')
    
champion_idx = np.argmax(accuracies)
bars[champion_idx].set_edgecolor('red')
bars[champion_idx].set_linewidth(3)

ax.set_title('Accuracy Comparison: Baseline vs WOA vs GA', fontsize=14, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_ylim([max(0, min(a * 100 for a in accuracies) - 5), 103])
ax.set_facecolor('#fdfdfd')

gold_patch = mpatches.Patch(facecolor='none', edgecolor='red', linewidth=3, label='Best Method')
ax.legend(handles=[gold_patch], fontsize=10, loc='upper right')

plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=1500, bbox_inches='tight')
plt.show()
print("Figure saved: accuracy_comparison.png")

In [ ]:
# Feature Selection Comparision (WOA vs GA)

fig, ax = plt.subplots(figsize=(12, 6), **FIG_STYLE)
x = np.arange(N_FEATURES)

bar_width  = 0.35
scale = 0.8

woa_bars = ax.bar(x - bar_width / 2, woa_feature_mask*scale,bar_width, label='WOA', color='#1565C0', alpha=0.85,edgecolor='black', linewidth=0.5)
ga_bars  = ax.bar(x + bar_width / 2, ga_feature_mask*scale,bar_width, label='GA',  color='#2E7D32', alpha=0.85,edgecolor='black', linewidth=0.5)

ax.set_ylim(0, scale + 0.2)
ax.set_xticks(x)
ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha='right', fontsize=9)
ax.set_yticks([0, scale])
ax.set_yticklabels(['Not Selected', 'Selected'], fontsize=10)
ax.set_title('Feature Selection: WOA vs GA', fontsize=14, fontweight='bold')

ax.set_ylabel('Selected (1) / Not Selected (0)', fontsize=11)
ax.legend(fontsize=11,loc='upper left')
ax.set_facecolor('#fdfdfd')

plt.tight_layout()
plt.savefig('feature_selection.png', dpi=1500, bbox_inches='tight')
plt.show()
print("Figure saved: feature_selection.png")


In [ ]:
# all 3 convergence in 1 graph

fig, ax = plt.subplots(figsize=(12, 6), **FIG_STYLE)

plt.style.use('fivethirtyeight')

ax.plot(range(len(woa_convergence)), woa_convergence,color='#1565C0', linewidth=4, linestyle = '--', label='WOA')
ax.plot(range(len(ga_convergence)),  ga_convergence,color='#2E7D32', linewidth=4, label='GA')
ax.axhline(y=baseline_accuracy, color='#B71C1C', linestyle='--', linewidth=2.5, label=f'Baseline KNN ({baseline_accuracy:.4f})')

ax.set_title('Convergence Comparison: WOA vs GA', fontsize=14, fontweight='bold')
ax.set_xlabel('Iteration / Generation', fontsize=12)
ax.set_ylabel('Best Accuracy', fontsize=12)
ax.legend(fontsize=11)
ax.set_facecolor('#fdfdfd')

plt.tight_layout()
plt.savefig('convergence_comparison.png', dpi=1500, bbox_inches='tight')
plt.show()
print("Figure saved: convergence_comparison.png")